[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/10-agentic-ai/notebooks/04-multi-agent-memory-mcp.ipynb)

# Notebook 04 — Crews, multi-agent patterns, memory and MCP

**Goal:** see that "multi-agent" frameworks are built from simple parts, add memory to an agent, and run a real **MCP** server and client.

1. A plain-Python mimic of **CrewAI** (agents with roles, tasks, a crew).
2. **Supervisor**, **handoff** and **debate** patterns.
3. **Short-term** and **long-term** memory.
4. **MCP**: define a server with tools, resources and prompts; connect a client; call the tools.

CrewAI is not installed here (it is large). The real CrewAI code is in `code/crewai_example.py`.

## Part 1 — A mini "crew" in plain Python

In CrewAI an **Agent** has a role, goal and backstory; a **Task** has a description, expected output and an agent; a **Crew** runs tasks in order, passing earlier outputs forward as context.

Below, each agent's *system prompt* is built from its role/goal/backstory exactly as a framework would. The LLM is a fake function that returns a canned reply per role, and prints the prompt it received so you can see what the model would see.

In [1]:
from dataclasses import dataclass, field

def fake_llm(system: str, user: str) -> str:
    """Stand-in for a chat model. Picks a canned answer based on the role in the system prompt."""
    if "Research Analyst" in system:
        return "1. MCP is an open protocol. 2. Hosts contain clients. 3. Servers expose tools, resources, prompts."
    if "Technical Writer" in system:
        return "MCP is like a USB-C port for AI apps: one standard plug so any app can use any tool server."
    return "OK."

@dataclass
class Agent:
    role: str
    goal: str
    backstory: str
    def system_prompt(self) -> str:
        return f"You are a {self.role}. Your goal: {self.goal}. Background: {self.backstory}"
    def work(self, task_text: str) -> str:
        return fake_llm(self.system_prompt(), task_text)

@dataclass
class Task:
    description: str
    expected_output: str
    agent: Agent

@dataclass
class Crew:
    agents: list
    tasks: list
    log: list = field(default_factory=list)
    def kickoff(self, inputs: dict) -> str:
        context = ""
        for t in self.tasks:                                   # sequential process
            prompt = t.description.format(**inputs) + f"\nExpected output: {t.expected_output}"
            if context:
                prompt += f"\nContext from previous task:\n{context}"
            print(f"\n>>> {t.agent.role} receives:\n{prompt}")
            context = t.agent.work(prompt)
            print(f"<<< {t.agent.role} returns:\n{context}")
            self.log.append((t.agent.role, context))
        return context

researcher = Agent("Research Analyst", "collect facts about {topic}", "careful and precise")
writer = Agent("Technical Writer", "explain {topic} to beginners", "loves analogies")
crew = Crew([researcher, writer], [
    Task("List 3 key facts about {topic}.", "a numbered list", researcher),
    Task("Write a one-sentence beginner explanation of {topic}.", "one sentence", writer),
])
print("\nFINAL:", crew.kickoff({"topic": "the Model Context Protocol"}))


>>> Research Analyst receives:
List 3 key facts about the Model Context Protocol.
Expected output: a numbered list
<<< Research Analyst returns:
1. MCP is an open protocol. 2. Hosts contain clients. 3. Servers expose tools, resources, prompts.

>>> Technical Writer receives:
Write a one-sentence beginner explanation of the Model Context Protocol.
Expected output: one sentence
Context from previous task:
1. MCP is an open protocol. 2. Hosts contain clients. 3. Servers expose tools, resources, prompts.
<<< Technical Writer returns:
MCP is like a USB-C port for AI apps: one standard plug so any app can use any tool server.

FINAL: MCP is like a USB-C port for AI apps: one standard plug so any app can use any tool server.


That is the core of a crew: **prompts built from roles + a loop over tasks + passing outputs forward**. Frameworks add tools, retries, delegation and logging on top.

## Part 2 — Supervisor, handoff and debate

**Supervisor:** a router decides which specialist handles a request. In real life the router is an LLM asked to reply with one word; here keyword rules stand in.

In [2]:
SPECIALISTS = {
    "billing": lambda r: "Billing agent: refund issued for the duplicate charge.",
    "tech":    lambda r: "Tech agent: clear the cache and log in again.",
    "general": lambda r: "General agent: a human will follow up.",
}

def supervisor(request: str) -> str:
    t = request.lower()
    choice = ("billing" if any(w in t for w in ["refund", "charge", "invoice"]) else
              "tech" if any(w in t for w in ["error", "login", "crash"]) else "general")
    print(f"[supervisor] '{request}' -> {choice}")
    return SPECIALISTS[choice](request)

for req in ["I was charged twice", "Login error 500", "Do you have stickers?"]:
    print("  ", supervisor(req))

[supervisor] 'I was charged twice' -> billing
   Billing agent: refund issued for the duplicate charge.
[supervisor] 'Login error 500' -> tech
   Tech agent: clear the cache and log in again.
[supervisor] 'Do you have stickers?' -> general
   General agent: a human will follow up.


**Handoff:** each agent updates a shared context and names the next agent (or `None` to stop). A **hop limit** prevents agents passing the baton back and forth forever.

In [3]:
def intake(ctx):   ctx["issue"] = "app crashes on login"; return ctx, "tech"
def tech(ctx):     ctx["fix"] = "update to v2.1";         return ctx, "followup"
def followup(ctx): ctx["reply"] = f"Please {ctx['fix']} to fix: {ctx['issue']}."; return ctx, None
AGENTS = {"intake": intake, "tech": tech, "followup": followup}

ctx, current, hops = {}, "intake", 0
while current and hops < 5:
    print("handoff ->", current)
    ctx, current = AGENTS[current](ctx)
    hops += 1
print(ctx)

handoff -> intake
handoff -> tech
handoff -> followup
{'issue': 'app crashes on login', 'fix': 'update to v2.1', 'reply': 'Please update to v2.1 to fix: app crashes on login.'}


**Debate:** two agents answer; a judge scores them against a **rubric** (a list of criteria). Real judges are LLMs given the rubric.

In [4]:
answers = {
    "agent_A": "Use an AI agent; it adapts if the page changes.",
    "agent_B": "Use a fixed script: cheaper, faster, and the same result every time; add an agent only if pages change often.",
}
rubric = ["cheaper", "faster", "same result", "only if"]
scores = {name: sum(word in text.lower() for word in rubric) for name, text in answers.items()}
print(scores, "-> winner:", max(scores, key=scores.get))

{'agent_A': 0, 'agent_B': 4} -> winner: agent_B


## Part 3 — Memory

**Short-term memory** = the last few messages. A `deque` with `maxlen` drops old ones automatically. Real systems also *summarise* what falls off.

In [5]:
from collections import deque
short_term = deque(maxlen=3)
for i in range(1, 6):
    short_term.append(f"user message {i}")
print(list(short_term))

['user message 3', 'user message 4', 'user message 5']


**Long-term memory** = facts stored as **embeddings** (lists of numbers capturing meaning) and recalled by **cosine similarity** (how closely two vectors point the same way). We use the small `all-MiniLM-L6-v2` model; if it cannot be downloaded, a simple word-hashing embedder is used instead so the notebook still runs.

In [6]:
import numpy as np

def get_embedder():
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        print("using MiniLM embeddings")
        return lambda texts: m.encode(texts, normalize_embeddings=True)
    except Exception as e:
        print("MiniLM unavailable, using hashing fallback:", type(e).__name__)
        def h(texts, dim=256):
            out = np.zeros((len(texts), dim))
            for i, t in enumerate(texts):
                for w in t.lower().split():
                    out[i, hash(w.strip(".,?!")) % dim] += 1
            return out / (np.linalg.norm(out, axis=1, keepdims=True) + 1e-9)
        return h

embed = get_embedder()
facts = ["The user's name is Priya.", "Priya prefers answers in Telugu.",
         "Priya's project deadline is 5 December.", "Priya is allergic to peanuts."]
fact_vecs = embed(facts)

def recall(query, k=1):
    scores = fact_vecs @ embed([query])[0]
    return [(round(float(scores[i]), 3), facts[i]) for i in np.argsort(-scores)[:k]]

for q in ["Which language should I reply in?", "When is the project due?", "Can she eat satay sauce?"]:
    print(q, "->", recall(q))

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18213.81it/s]

using MiniLM embeddings
Which language should I reply in? -> [(0.367, 'Priya prefers answers in Telugu.')]
When is the project due? -> [(0.595, "Priya's project deadline is 5 December.")]


Can she eat satay sauce? -> [(0.283, 'Priya is allergic to peanuts.')]


An agent uses long-term memory by recalling relevant facts and **putting them into the prompt** before calling the LLM:

In [7]:
def build_prompt(user_msg):
    memories = [f for _, f in recall(user_msg, k=2)]
    history = "\n".join(short_term)
    return f"Known facts: {memories}\nRecent chat:\n{history}\nUser: {user_msg}"
print(build_prompt("Remind me when my deadline is"))

Known facts: ["Priya's project deadline is 5 December.", "The user's name is Priya."]
Recent chat:
user message 3
user message 4
user message 5
User: Remind me when my deadline is


## Part 4 — MCP: a real server and client

We define an MCP server with the official `mcp` package. In **mcp 2.x** the class is `MCPServer`; in **1.x** it was `FastMCP`. The cell installs `mcp` if missing (e.g. in Colab) and imports whichever exists.

* `@server.tool()` → a **tool** the model may call.
* `@server.resource("notes://...")` → a **resource**: read-only data addressed by a URI.
* `@server.prompt()` → a **prompt** template the user can pick.

In [8]:
try:
    import mcp
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp"], check=True)
from importlib.metadata import version
print("mcp", version("mcp"))
try:
    from mcp.server.mcpserver import MCPServer
except ImportError:
    from mcp.server.fastmcp import FastMCP as MCPServer

server = MCPServer("study-helper")
NOTES = {"agents": "An agent is an LLM in a loop with tools.",
         "mcp": "MCP connects AI apps to tool servers in one standard way.",
         "react": "ReAct = Reason + Act."}

@server.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@server.tool()
def search_notes(keyword: str) -> str:
    """Return study notes containing the keyword."""
    hits = [f"{k}: {v}" for k, v in NOTES.items() if keyword.lower() in (k + v).lower()]
    return "\n".join(hits) or "no match"

@server.resource("notes://index")
def notes_index() -> str:
    """All note topics."""
    return ", ".join(NOTES)

@server.resource("notes://{topic}")
def note(topic: str) -> str:
    """One note by topic."""
    return NOTES.get(topic, "no such note")

@server.prompt()
def quiz_me(topic: str) -> str:
    """Ask the model to quiz me."""
    return f"Ask me 3 short quiz questions about {topic}, one at a time."
print("server defined:", server.name)

mcp 2.3.0
server defined: study-helper


Now connect a **client**, exactly as a host app (Claude Desktop, VS Code) would: list what the server offers, then call a tool, read a resource and fetch a prompt.

In mcp 2.x, `mcp.Client(server)` connects **in memory** (no subprocess) — perfect for testing. MCP is **asynchronous** (it waits on input/output), so we use `async`/`await`; Jupyter lets us `await` directly in a cell. If an older SDK without `Client` is installed, we call the server's own methods instead.

In [9]:
async def explore():
    try:
        from mcp import Client
    except ImportError:
        Client = None
    if Client is None:                                   # mcp 1.x fallback
        tools = await server.list_tools()
        print("Tools:", [t.name for t in tools])
        print(await server.call_tool("add", {"a": 2, "b": 40}))
        return
    async with Client(server) as client:
        tools = (await client.list_tools()).tools
        print("Tools:", [t.name for t in tools])
        print("Schema of add:", tools[0].input_schema)
        r = await client.call_tool("add", {"a": 2, "b": 40})
        print("add ->", r.content[0].text)
        r = await client.call_tool("search_notes", {"keyword": "loop"})
        print("search_notes ->", r.content[0].text)
        print("Resources:", [x.uri for x in (await client.list_resources()).resources])
        print("Templates:", [x.uri_template for x in (await client.list_resource_templates()).resource_templates])
        print("notes://react ->", (await client.read_resource("notes://react")).contents[0].text)
        p = await client.get_prompt("quiz_me", {"topic": "MCP"})
        print("prompt ->", p.messages[0].content.text)

await explore()

Tools: ['add', 'search_notes']
Schema of add: {'type': 'object', 'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'addArguments'}
add -> 42.0
search_notes -> agents: An agent is an LLM in a loop with tools.
Resources: ['notes://index']
Templates: ['notes://{topic}']
notes://react -> ReAct = Reason + Act.
prompt -> Ask me 3 short quiz questions about MCP, one at a time.


### Connecting over stdio to the script in `code/`

Real hosts start the server as a separate process and talk over **stdio** (its standard input/output). If you are running inside the repo, this cell starts `../code/mcp_server.py` that way. In Colab the file is not there, so the cell skips.

In [10]:
import sys, pathlib
script = pathlib.Path("../code/mcp_server.py")
try:
    from mcp import Client, StdioServerParameters
    has_client = True
except ImportError:
    has_client = False

if script.exists() and has_client:
    async def over_stdio():
        params = StdioServerParameters(command=sys.executable, args=[str(script.resolve())])
        async with Client(params) as c:
            print("stdio tools:", [t.name for t in (await c.list_tools()).tools])
            print((await c.call_tool("search_notes", {"keyword": "protocol"})).content[0].text)
    await over_stdio()
else:
    print("code/mcp_server.py not found here (or old mcp) - skipping the stdio demo.")

stdio tools: ['add', 'search_notes', 'save_note']
mcp: MCP is an open protocol that connects AI apps to tools and data in a standard way.


### From MCP tools to model tools

A host turns each MCP tool into a tool definition for its model. The MCP `input_schema` *is* JSON Schema, so it drops straight into the Anthropic format from notebook 01:

In [11]:
async def mcp_tools_as_anthropic():
    from mcp import Client
    async with Client(server) as c:
        return [{"name": t.name, "description": t.description, "input_schema": t.input_schema}
                for t in (await c.list_tools()).tools]
try:
    import json
    print(json.dumps(await mcp_tools_as_anthropic(), indent=2))
except ImportError:
    print("needs mcp 2.x Client")

[
  {
    "name": "add",
    "description": "Add two numbers.",
    "input_schema": {
      "type": "object",
      "properties": {
        "a": {
          "title": "A",
          "type": "number"
        },
        "b": {
          "title": "B",
          "type": "number"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "title": "addArguments"
    }
  },
  {
    "name": "search_notes",
    "description": "Return study notes containing the keyword.",
    "input_schema": {
      "type": "object",
      "properties": {
        "keyword": {
          "title": "Keyword",
          "type": "string"
        }
      },
      "required": [
        "keyword"
      ],
      "title": "search_notesArguments"
    }
  }
]


## Recap

* A **crew** = role-based system prompts + a loop over tasks passing outputs forward.
* **Supervisor / handoff / debate** are small control-flow patterns around LLM calls.
* **Memory** = what you put back into the prompt: recent messages + recalled facts.
* **MCP** = one standard way for hosts to discover and call tools, read resources and use prompts from any server.

**Try it:** add a `word_count(text: str) -> int` tool to the MCP server, re-run the client cell, and check it appears with the right schema.